# RFdiffusion — BU SCC

Run this notebook **inside an SCC GPU allocation**, using SCC OnDemand Jupyter or
Jupyter on an allocated compute node. Opening `qrsh` in a separate terminal does
not give an existing login-node notebook access to that GPU.

This follows the workflow of your `BindCraft_scc.ipynb`: configure paths and design
settings, validate, check the GPU, start a background run, monitor/stop it, and inspect results.
It reuses SE3nv and existing weights. It does not install packages or submit a separate batch job.
Both start and stop flags are **False** in the saved notebook.

hostname -f

python -m notebook --no-browser --ip="$(hostname -f)" --port=8888

## Start an SCC session

Recommended: open [SCC OnDemand](https://scc-ondemand.bu.edu) and request a GPU Jupyter
session for project `dunlop`, four CPU cores, and sufficient wall time.
For the repository's legacy CUDA 11.1 / PyTorch 1.9 stack, start with P100 or V100;
a newer GPU is not automatically compatible. Check resources with `qgpus -s -v`.

Alternatively, request an interactive allocation from an SCC terminal:

```bash
qrsh -P dunlop -l gpus=1 -l gpu_type=P100 -pe omp 4 -l h_rt=04:00:00 -now n
```

After it starts, launch Jupyter using an already installed Jupyter environment
and SCC's SSH forwarding instructions. The notebook kernel needs Python 3 and
IPython; scientific subprocesses use the separate `ENV_PREFIX/bin/python` below.
OnDemand handles browser access for you. Do not install Jupyter into SE3nv just
for this notebook.

The GPU fix is persistent: you do not reinstall it each session. Preflight checks
the existing environment against the allocated GPU. The earlier installation audit
found CPU-only PyTorch; if that is still present, follow the manual repair section
in [the installation and batch guide](01_running_rfdiffusion_on_scc.ipynb).

Sources: [BU GPU computing](https://www.bu.edu/tech/support/research/software-and-programming/gpu-computing/),
[BU interactive jobs](https://www.bu.edu/tech/support/research/system-usage/running-jobs/interactive-jobs/),
and the [RFdiffusion README](../README.md).


In [1]:
import html
import json
import os
import re
import socket
import subprocess
import sys
import textwrap
import uuid
from datetime import datetime
from pathlib import Path
from IPython.display import HTML, display

REPO_ROOT = Path('/project/dunlop/fereshteh/RFdiffusion/RFdiffusion')
ENV_PREFIX = Path('/projectnb/dunlop/fereshteh/.conda/envs/SE3nv')
PROJECT_WORK = Path('/projectnb/dunlop/fereshteh/RFdiffusion')
PYTHON = ENV_PREFIX / 'bin/python'
LOCAL_STATE = PROJECT_WORK / '.rfdiffusion-scc'

def local_path(value):
    path = Path(value).expanduser()
    return (path if path.is_absolute() else REPO_ROOT / path).resolve()

def require_gpu_allocation():
    if not sys.platform.startswith('linux') or not os.environ.get('JOB_ID'):
        raise RuntimeError('Start this notebook kernel inside an SCC GPU allocation.')
    visible = os.environ.get('CUDA_VISIBLE_DEVICES', '').strip()
    if not visible or visible in {'-1', 'NoDevFiles'}:
        raise RuntimeError('No assigned GPU. Request a GPU session; do not set CUDA_VISIBLE_DEVICES manually.')

def compute_env():
    env = dict(os.environ)
    env.update(CONDA_PREFIX=str(ENV_PREFIX),
               PATH=str(ENV_PREFIX / 'bin') + ':' + env.get('PATH', ''),
               PYTHONNOUSERSITE='1', PYTHONDONTWRITEBYTECODE='1',
               PYTHONUNBUFFERED='1', DGLBACKEND='pytorch')
    for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
        env[key] = env.get('NSLOTS', '1')
    return env  # Preserve scheduler CUDA visibility and module library paths.

def run_python(source, args=(), timeout=180):
    result = subprocess.run([str(PYTHON), '-u', '-', *map(str, args)],
                            input=textwrap.dedent(source), cwd=REPO_ROOT,
                            env=compute_env(), text=True, capture_output=True, timeout=timeout)
    if result.returncode:
        raise RuntimeError(result.stdout + result.stderr)
    if result.stderr.strip():
        print(result.stderr.strip())
    return result.stdout

assert (REPO_ROOT / 'scripts/run_inference.py').is_file(), REPO_ROOT
assert PYTHON.is_file(), PYTHON
PREFLIGHT_OK = False
print('Kernel host:', socket.gethostname(), '| Job:', os.environ.get('JOB_ID', 'none'))
print('Assigned GPUs:', os.environ.get('CUDA_VISIBLE_DEVICES', 'none'))
print('Compute Python:', PYTHON)
print('Results and records:', PROJECT_WORK)


Kernel host: scc-c11 | Job: 7726301
Assigned GPUs: 0
Compute Python: /projectnb/dunlop/fereshteh/.conda/envs/SE3nv/bin/python
Results and records: /projectnb/dunlop/fereshteh/RFdiffusion


## Design settings

Start with **one 50-residue unconditional design** to verify the pipeline.
Then select `binder` or `motif` and use the supplied example paths, or substitute your target.
Relative input paths resolve from the clone. Output paths use persistent project storage.
Each launch creates a unique subdirectory; it never resumes or overwrites an earlier run.
An interrupted diffusion trajectory cannot be resumed by this notebook.

The notebook time limit is separate from the allocation's wall time. Keep the GPU
Jupyter session alive during computation. Changing settings requires validation and preflight again.


In [2]:
MODE = 'unconditional'  # unconditional, binder, motif
RUN_NAME = 'trial'
NUM_DESIGNS = 1
MAX_WALL_MINUTES = 60  # None removes only the notebook limit.
WRITE_TRAJECTORY = True

# Unconditional design: sampled length between these bounds (inclusive).
LENGTH_RANGE = [50, 50]

# Binder example from examples/design_ppi.sh; edit all three for your target.
BINDER_PDB = 'examples/input_pdbs/insulin_target.pdb'
BINDER_CONTIG = 'A1-150/0 70-100'
HOTSPOTS = ['A59', 'A83', 'A91']

# Motif example from examples/design_motifscaffolding.sh.
MOTIF_PDB = 'examples/input_pdbs/5TPN.pdb'
MOTIF_CONTIG = '10-40/A163-181/10-40'

# Optional checkpoint override, e.g. models/ActiveSite_ckpt.pt for small motifs.
CHECKPOINT_OVERRIDE = ''


## Validate inputs and preview the command

`A163-181` means a fixed input PDB segment; `10-40` means newly generated residues.
`/0 ` (with the space) introduces a chain break. The binder example preserves target
A1–150 and generates a separate 70–100-residue binder. Hotspots identify target residues
where contact is encouraged. The validator checks the simple contig syntax used here,
input CA residue records, and hotspot membership in the target contig.

For partial diffusion, symmetry, or fold conditioning, use the upstream examples or
extend the batch guide deliberately; these presets cover the three basic tasks only.


In [3]:
def settings_signature():
    return json.dumps(dict(mode=MODE, name=RUN_NAME, count=NUM_DESIGNS,
        minutes=MAX_WALL_MINUTES, trajectory=WRITE_TRAJECTORY, length=LENGTH_RANGE,
        binder=[BINDER_PDB, BINDER_CONTIG, HOTSPOTS], motif=[MOTIF_PDB, MOTIF_CONTIG],
        checkpoint=CHECKPOINT_OVERRIDE, repo=str(REPO_ROOT), env=str(ENV_PREFIX),
        work=str(PROJECT_WORK)), sort_keys=True)

def validate_settings():
    if MODE not in {'unconditional', 'binder', 'motif'}:
        raise ValueError('Choose unconditional, binder, or motif.')
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]*', RUN_NAME):
        raise ValueError('Use a simple RUN_NAME containing letters, numbers, dots, _ or -.')
    if type(NUM_DESIGNS) is not int or NUM_DESIGNS < 1:
        raise ValueError('NUM_DESIGNS must be a positive integer.')
    if MAX_WALL_MINUTES is not None and (type(MAX_WALL_MINUTES) not in (int, float)
            or not 0 < MAX_WALL_MINUTES < float('inf')):
        raise ValueError('MAX_WALL_MINUTES must be positive and finite, or None.')
    if type(WRITE_TRAJECTORY) is not bool:
        raise ValueError('WRITE_TRAJECTORY must be True or False.')
    args = [f'inference.num_designs={NUM_DESIGNS}',
            f'inference.write_trajectory={str(WRITE_TRAJECTORY).lower()}']
    target = None
    if MODE == 'unconditional':
        if not (len(LENGTH_RANGE) == 2 and all(type(x) is int and x > 0 for x in LENGTH_RANGE)
                and LENGTH_RANGE[0] <= LENGTH_RANGE[1]):
            raise ValueError('LENGTH_RANGE must be [positive minimum, maximum].')
        contig = f'{LENGTH_RANGE[0]}-{LENGTH_RANGE[1]}'
    else:
        target = local_path(BINDER_PDB if MODE == 'binder' else MOTIF_PDB)
        if not target.is_file():
            raise FileNotFoundError(target)
        residues = set()
        insertion_residues = set()
        for line in target.read_text().splitlines():
            if line.startswith('ENDMDL'):
                break
            if line.startswith('ATOM  ') and line[12:16].strip() == 'CA':
                if line[26].strip():
                    insertion_residues.add((line[21], int(line[22:26])))
                residues.add((line[21], int(line[22:26])))
        contig = BINDER_CONTIG if MODE == 'binder' else MOTIF_CONTIG
        fixed = set()
        for token in re.split(r'[/ ]+', contig.strip()):
            match = re.fullmatch(r'([A-Za-z])([0-9]+)(?:-([0-9]+))?', token)
            if match:
                chain, first, last = match.groups()
                first, last = int(first), int(last or first)
                if first > last:
                    raise ValueError('Reversed residue range: ' + token)
                fixed.update((chain, i) for i in range(first, last + 1))
            elif not re.fullmatch(r'[0-9]+(?:-[0-9]+)?', token):
                raise ValueError('Unsupported contig token: ' + token)
            elif token != '0':
                bounds = list(map(int, token.split('-')))
                if min(bounds) < 1 or bounds[0] > bounds[-1]:
                    raise ValueError('Invalid generated length: ' + token)
        if fixed & insertion_residues:
            raise ValueError('Selected contig includes insertion codes; renumber those input residues first.')
        if not fixed or fixed - residues:
            raise ValueError('Missing fixed input residues: ' + str(sorted(fixed - residues)[:15]))
        if MODE == 'binder':
            if '/0 ' not in contig or not HOTSPOTS:
                raise ValueError('Binder mode requires a /0 space chain break and target hotspots.')
            for hotspot in HOTSPOTS:
                match = re.fullmatch(r'([A-Za-z])([0-9]+)', hotspot)
                if not match or (match[1], int(match[2])) not in fixed:
                    raise ValueError('Hotspot must be in the fixed target contig: ' + hotspot)
            args += ['ppi.hotspot_res=[' + ','.join(HOTSPOTS) + ']',
                     'denoiser.noise_scale_ca=0', 'denoiser.noise_scale_frame=0']
        args += ['inference.input_pdb=' + json.dumps(str(target))]
    args += ['contigmap.contigs=[' + contig + ']']
    checkpoint = local_path(CHECKPOINT_OVERRIDE) if CHECKPOINT_OVERRIDE else REPO_ROOT / 'models' / (
        'Complex_base_ckpt.pt' if MODE == 'binder' else 'Base_ckpt.pt')
    if not checkpoint.is_file() or not checkpoint.stat().st_size:
        raise FileNotFoundError(checkpoint)
    args += ['inference.ckpt_override_path=' + json.dumps(str(checkpoint))]
    return dict(overrides=args, checkpoint=str(checkpoint), target=None if target is None else str(target))

PREFLIGHT_OK = False
SETTINGS = validate_settings()
VALIDATED_SIGNATURE = settings_signature()
print(json.dumps(SETTINGS, indent=2))


{
  "overrides": [
    "inference.num_designs=1",
    "inference.write_trajectory=true",
    "contigmap.contigs=[50-50]",
    "inference.ckpt_override_path=\"/project/dunlop/fereshteh/RFdiffusion/RFdiffusion/models/Base_ckpt.pt\""
  ],
  "checkpoint": "/project/dunlop/fereshteh/RFdiffusion/RFdiffusion/models/Base_ckpt.pt",
  "target": null
}


## GPU preflight

This runs in SE3nv on the kernel's allocated node. It checks the actual RFdiffusion
imports, a PyTorch GPU operation, DGL GPU message passing, and loads the selected
checkpoint on CPU. It does not generate a design. Passing this small check does not
guarantee enough GPU memory for a large target; the first design is the end-to-end test.

If PyTorch reports no compiled CUDA runtime, use the one-time repair instructions in
the other notebook. Do not reinstall packages merely because a new session has started.


In [4]:
PREFLIGHT_OK = False
require_gpu_allocation()
if settings_signature() != globals().get('VALIDATED_SIGNATURE'):
    raise RuntimeError('Rerun input validation after editing settings.')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=30).stdout)
preflight_output = run_python(r"""
import sys
import torch, dgl, e3nn
import dgl.function as fn
from se3_transformer.model import SE3Transformer
from rfdiffusion.inference import model_runners
print('Python:', sys.executable)
print('torch:', torch.__version__, 'CUDA runtime:', torch.version.cuda)
print('DGL:', dgl.__version__, 'e3nn:', e3nn.__version__)
assert torch.version.cuda is not None, 'CPU-only PyTorch: follow the one-time GPU repair guide.'
assert torch.cuda.is_available(), 'No usable GPU in this allocation.'
a = torch.ones((8, 8), device='cuda')
assert (a @ a)[0, 0].item() == 8
print('GPU:', torch.cuda.get_device_name(0))
graph = dgl.graph(([0, 1], [1, 0])).to('cuda')
graph.ndata['x'] = torch.ones((2, 1), device='cuda')
graph.update_all(fn.copy_u('x', 'm'), fn.sum('m', 'y'))
assert torch.allclose(graph.ndata['y'], graph.ndata['x'])
torch.cuda.synchronize()
checkpoint = torch.load(sys.argv[1], map_location='cpu')
assert 'model_state_dict' in checkpoint and 'config_dict' in checkpoint
print('Checkpoint loaded:', sys.argv[1])
print('RFdiffusion compute preflight passed.')
""", [SETTINGS['checkpoint']], timeout=240)
print(preflight_output)
PREFLIGHT_SIGNATURE = settings_signature()
PREFLIGHT_OK = True


Thu Sep 24 15:22:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-12GB           On  |   00000000:02:00.0 Off |                    0 |
| N/A   35C    P0             25W /  250W |       0MiB /  12288MiB |      0%   E. Process |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Start a run inside this allocation

Set `RUN_DESIGN = True` after preflight passes. Each launch uses a unique directory
under `PROJECT_WORK/results/`, plus a settings record, log and state under
`PROJECT_WORK/.rfdiffusion-scc/runs/`. The supervisor enforces the notebook time limit
and stops the entire inference process group when requested. Only one active run is
allowed from this notebook kernel; the status/stop cells let you manage it.

This is not a `qsub` job: it runs on this allocated node and depends on the allocation
remaining alive. For jobs that should outlive your Jupyter session, use the separate
[batch guide](01_running_rfdiffusion_on_scc.ipynb). A hard scheduler kill can leave
stale state; always check the actual SCC job before assuming a process is running.


In [5]:
SUPERVISOR_SOURCE = r"""
import json, os, signal, socket, subprocess, sys, time
from pathlib import Path
run_dir = Path(sys.argv[1])
launch = json.loads((run_dir / 'launch.json').read_text())
state_path = run_dir / 'state.json'
lock_path = Path(launch['output']) / '.rfdiffusion-notebook.lock'
process = None
locked = False
def save_state(**changes):
    state.update(changes)
    temp = state_path.with_suffix('.tmp')
    temp.write_text(json.dumps(state, indent=2))
    temp.replace(state_path)
def stop(signum, frame):
    raise KeyboardInterrupt
def terminate_design():
    if process is not None and process.poll() is None:
        os.killpg(process.pid, signal.SIGTERM)
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
signal.signal(signal.SIGTERM, stop)
signal.signal(signal.SIGINT, stop)
state = dict(hostname=socket.gethostname(), job_id=os.environ.get('JOB_ID'), status='starting', supervisor_pid=os.getpid(), run_dir=str(run_dir))
save_state()
try:
    # Exclusive lock avoids mixing simultaneous campaigns in the same CSV files.
    with lock_path.open('x') as lock:
        json.dump(dict(supervisor_pid=os.getpid(), run_dir=str(run_dir)), lock)
    locked = True
    argv = launch['argv']
    print('Launching:', argv, flush=True)
    process = subprocess.Popen(argv, cwd=launch['root'], stdin=subprocess.DEVNULL,
                               start_new_session=True)
    save_state(status='running', design_pid=process.pid, started_at=time.time())
    timeout = None if launch['max_wall_minutes'] is None else launch['max_wall_minutes'] * 60
    code = process.wait(timeout=timeout)
    save_state(status='finished' if code == 0 else 'failed', returncode=code)
except subprocess.TimeoutExpired:
    terminate_design()
    save_state(status='time_limit', returncode=process.returncode)
    print('Stopped at the notebook wall-time limit.', flush=True)
except KeyboardInterrupt:
    terminate_design()
    save_state(status='stopped', returncode=None if process is None else process.returncode)
    print('Stopped by notebook request.', flush=True)
except BaseException as exc:
    terminate_design()
    save_state(status='failed', error=repr(exc))
    raise
finally:
    if locked:
        lock_path.unlink(missing_ok=True)
    save_state(ended_at=time.time())
"""

RUN_DESIGN = True
if RUN_DESIGN:
    require_gpu_allocation()
    if not globals().get('PREFLIGHT_OK', False) or settings_signature() != globals().get('PREFLIGHT_SIGNATURE'):
        raise RuntimeError('Validate and pass GPU preflight for these settings first.')
    if globals().get('ACTIVE_SUPERVISOR') is not None and ACTIVE_SUPERVISOR.poll() is None:
        raise RuntimeError('This kernel already has a running design. Use the stop cell first.')
    # Recompute validated arguments, rather than trusting a possibly edited SETTINGS dictionary.
    launch_settings = validate_settings()
    run_name = RUN_NAME + '_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
    RUN_DIR = LOCAL_STATE / 'runs' / run_name
    DESIGN_DIR = PROJECT_WORK / 'results' / run_name
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    DESIGN_DIR.mkdir(parents=True, exist_ok=False)
    # Isolate schedule cache writes from concurrent notebook runs.
    schedule_dir = RUN_DIR / 'schedules'
    schedule_dir.mkdir()
    # Reuse any completed upstream tables without sharing new cache writes.
    import shutil
    for cached in (REPO_ROOT / 'schedules').glob('*.pkl'):
        shutil.copy2(cached, schedule_dir / cached.name)
    argv = [str(PYTHON), '-u', str(REPO_ROOT / 'scripts/run_inference.py'),
            *launch_settings['overrides'],
            'inference.output_prefix=' + json.dumps(str(DESIGN_DIR / 'design')),
            'inference.schedule_directory_path=' + json.dumps(str(schedule_dir))]
    launch = dict(root=str(REPO_ROOT), output=str(DESIGN_DIR), argv=argv,
                  max_wall_minutes=MAX_WALL_MINUTES)
    (RUN_DIR / 'launch.json').write_text(json.dumps(launch, indent=2))
    (RUN_DIR / 'settings.json').write_text(settings_signature())
    (RUN_DIR / 'preflight.txt').write_text(preflight_output)
    (RUN_DIR / 'supervisor.py').write_text(SUPERVISOR_SOURCE)
    with (RUN_DIR / 'run.log').open('a') as log:
        ACTIVE_SUPERVISOR = subprocess.Popen(
            [str(PYTHON), '-u', str(RUN_DIR / 'supervisor.py'), str(RUN_DIR)],
            cwd=REPO_ROOT, env=compute_env(), stdin=subprocess.DEVNULL,
            stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    print('Supervisor PID:', ACTIVE_SUPERVISOR.pid)
    print('Run records:', RUN_DIR)
    print('Designs:', DESIGN_DIR)
else:
    print('Not started. Set RUN_DESIGN = True after passing preflight.')


Supervisor PID: 2132213
Run records: /projectnb/dunlop/fereshteh/RFdiffusion/.rfdiffusion-scc/runs/trial_20260924_152338_77a613c1
Designs: /projectnb/dunlop/fereshteh/RFdiffusion/results/trial_20260924_152338_77a613c1


## Status and stop controls

Rerun status to refresh the last part of the log. After restarting the kernel,
set `RUN_TO_INSPECT` to the printed run-record directory. Status files can be stale
after an allocation expires. Stop only from the original host and allocation; the
stop cell checks both and verifies the supervisor command before signaling its PID.


In [12]:
# Re-run this cell for progress. RUN_TO_INSPECT can select a run after a kernel restart.
RUN_TO_INSPECT = ""  # Absolute path to a run directory printed by the start cell.
if RUN_TO_INSPECT:
    RUN_DIR = local_path(RUN_TO_INSPECT)
elif "RUN_DIR" not in globals():
    candidates = sorted((LOCAL_STATE / "runs").glob("*/launch.json"))
    RUN_DIR = candidates[-1].parent if candidates else None
if RUN_DIR is None:
    print("No RFdiffusion notebook run found.")
else:
    print("Inspecting:", RUN_DIR)
    state_path = RUN_DIR / "state.json"
    print(state_path.read_text(encoding="utf-8") if state_path.exists() else "Starting; check again shortly.")
    log_path = RUN_DIR / "run.log"
    if log_path.exists():
        with log_path.open("rb") as log:
            log.seek(max(0, log_path.stat().st_size - 12000))
            print(log.read().decode("utf-8", errors="replace"))


Inspecting: /projectnb/dunlop/fereshteh/RFdiffusion/.rfdiffusion-scc/runs/trial_20260924_152338_77a613c1
{
  "hostname": "scc-c11",
  "job_id": "7726301",
  "status": "finished",
  "supervisor_pid": 2132213,
  "run_dir": "/projectnb/dunlop/fereshteh/RFdiffusion/.rfdiffusion-scc/runs/trial_20260924_152338_77a613c1",
  "design_pid": 2132214,
  "started_at": 1790277818.4978838,
  "returncode": 0,
  "ended_at": 1790277862.853643
}
iffusion on Tesla P100-PCIE-12GB
Reading models from /project/dunlop/fereshteh/RFdiffusion/RFdiffusion/rfdiffusion/inference/../../models
[2026-09-24 15:23:41,397][rfdiffusion.inference.model_runners][INFO] - Reading checkpoint from /project/dunlop/fereshteh/RFdiffusion/RFdiffusion/models/Base_ckpt.pt
This is inf_conf.ckpt_path
/project/dunlop/fereshteh/RFdiffusion/RFdiffusion/models/Base_ckpt.pt
Assembling -model, -diffuser and -preprocess configs from checkpoint
USING MODEL CONFIG: self._conf[model][n_extra_block] = 4
USING MODEL CONFIG: self._conf[model][n_mai

In [ ]:
# Select the desired run in the status cell, then set this flag to stop it.
STOP_RUN = False
if STOP_RUN:
    if not globals().get("RUN_DIR"):
        raise RuntimeError("Select a run using the status cell first.")
    print(run_python(r"""
import json, os, signal, socket, sys
from pathlib import Path
run_dir = Path(sys.argv[1])
state = json.loads((run_dir / 'state.json').read_text())
if state.get('hostname') != socket.gethostname() or state.get('job_id') != os.environ.get('JOB_ID'):
    raise RuntimeError('Stop from the original allocated host and job. No signal sent.')
pid = state['supervisor_pid']
try:
    argv = (Path('/proc') / str(pid) / 'cmdline').read_bytes().split(b'\0')
except FileNotFoundError:
    print('The supervisor has already exited.')
else:
    # Verify the unique run path before signaling; never signal an unrelated reused PID.
    if str(run_dir / 'supervisor.py').encode() not in argv:
        raise RuntimeError('PID does not belong to this run; no signal sent.')
    os.kill(pid, signal.SIGTERM)
    print('Stop requested. Allow up to 15 seconds, then refresh status.')
""", [str(RUN_DIR)]))
else:
    print("No stop requested.")


## Inspect completed backbone designs

The table lists final PDB/TRB pairs and residue counts. Wait until inference stops
before using outputs downstream: the PDB is written before its TRB metadata.
Use `RESULTS_TO_VIEW` to inspect another output directory. This does not rank designs
by binding affinity or produce BindCraft's accepted-design metrics.

- `.pdb`: generated backbone; designed positions are generally glycine placeholders.
- `.trb`: run configuration, mappings and per-step model confidence.
- `traj/`: intermediate trajectory PDBs when enabled.

Sequence design (such as ProteinMPNN), structure validation and binder assessment
are separate downstream work. PDB B-factors in this RFdiffusion script indicate
fixed versus diffused positions; they are **not pLDDT confidence scores**.


In [13]:
RESULTS_TO_VIEW = ''
if RESULTS_TO_VIEW:
    RESULTS_DIR = local_path(RESULTS_TO_VIEW)
elif globals().get('RUN_DIR') and (RUN_DIR / 'launch.json').is_file():
    RESULTS_DIR = Path(json.loads((RUN_DIR / 'launch.json').read_text())['output'])
else:
    RESULTS_DIR = None
completed_pdbs = []
if RESULTS_DIR is None:
    print('Select a run in the status cell or set RESULTS_TO_VIEW.')
else:
    if (RESULTS_DIR / '.rfdiffusion-notebook.lock').exists():
        print('Run lock present: inference may still be running. Counts below are provisional.')
    rows = []
    for pdb in sorted(RESULTS_DIR.glob('design_*.pdb')):
        chains = {}
        for line in pdb.read_text().splitlines():
            if line.startswith('ATOM  ') and line[12:16].strip() == 'CA':
                chains[line[21]] = chains.get(line[21], 0) + 1
        has_trb = pdb.with_suffix('.trb').is_file()
        if has_trb:
            completed_pdbs.append(pdb)
        rows.append([pdb.name, str(chains), 'PDB + TRB' if has_trb else 'PDB only'])
    if rows:
        table = '<table><tr><th>Design</th><th>CA residues by chain</th><th>Files</th></tr>'
        table += ''.join('<tr>' + ''.join('<td>' + html.escape(x) + '</td>' for x in row) + '</tr>' for row in rows)
        display(HTML(table + '</table>'))
    else:
        print('No final backbone PDB files yet:', RESULTS_DIR)


Design,CA residues by chain,Files
design_0.pdb,{'A': 50},PDB + TRB


## View a backbone

Select a completed PDB, or leave the path blank to show the first complete pair.
This viewer uses browser-side 3Dmol.js without installing Python visualization packages.
The browser must trust the notebook and reach the 3Dmol JavaScript CDN. If the view is
blank, open the PDB in PyMOL or ChimeraX. Chains use different colors.


In [14]:
PDB_TO_VIEW = ''
view_path = local_path(PDB_TO_VIEW) if PDB_TO_VIEW else (completed_pdbs[0] if completed_pdbs else None)
if view_path is None:
    print('No completed structure to display.')
else:
    pdb_text = view_path.read_text()
    viewer_id = 'rfd_' + uuid.uuid4().hex
    # Prevent a PDB remark containing </script> from ending the embedded script.
    pdb_json = json.dumps(pdb_text).replace('<', r'\u003c')
    viewer_html = f"""
    <div id="{viewer_id}" style="width:800px;height:500px;position:relative"></div>
    <script>
    (function() {{
      function draw() {{
        const viewer = $3Dmol.createViewer(document.getElementById('{viewer_id}'), {{backgroundColor:'white'}});
        viewer.addModel({pdb_json}, 'pdb');
        viewer.setStyle({{}}, {{cartoon:{{color:'spectrum'}}}});
        viewer.setStyle({{chain:'A'}}, {{cartoon:{{color:'#3c5b6f'}}}});
        viewer.setStyle({{chain:'B'}}, {{cartoon:{{color:'#B76E79'}}}});
        viewer.zoomTo(); viewer.render();
      }}
      if (window.$3Dmol) {{ draw(); }} else {{
        const script = document.createElement('script');
        script.src = 'https://3Dmol.org/build/3Dmol-min.js';
        script.onload = draw;
        script.onerror = () => {{ document.getElementById('{viewer_id}').textContent = 'Viewer CDN unavailable; open the PDB in a desktop viewer.'; }};
        document.head.appendChild(script);
      }}
    }})();
    </script>"""
    print(view_path)
    display(HTML(viewer_html))


/projectnb/dunlop/fereshteh/RFdiffusion/results/trial_20260924_152338_77a613c1/design_0.pdb


## SCC troubleshooting

- **No allocation:** the Jupyter kernel itself must run on the allocated GPU node.
  Use OnDemand or start Jupyter after `qrsh` grants the allocation.
- **CPU-only PyTorch:** use the one-time repair instructions in the companion guide.
  Loading a CUDA module does not convert CPU-only PyTorch into a CUDA build.
- **GPU kernel/library errors:** check GPU model, driver, PyTorch and DGL CUDA versions.
  Preflight checks both PyTorch and DGL operations; importing them alone is insufficient.
- **Target or hotspot errors:** check actual PDB chain letters and residue numbers.
  Prepare structures without ambiguous insertion codes and preserve the `/0 ` space.
- **Checkpoint missing:** check `models/`. The earlier audit found the optional epoch-8
  checkpoint under `Base_epoch8_ckpt.pt.1`; that exact path can be used as an override.
- **Calculating IGSO3:** the first uncached schedule can take time. The notebook copies
  existing tables into a per-run cache to avoid simultaneous cache writes across runs.
- **Out of memory:** begin with the short unconditional test and one design. Larger
  targets need more memory; a tiny GPU preflight is not a full inference memory test.
- **Stopped or timed out:** completed PDB/TRB pairs remain; an interrupted trajectory
  is not resumable. Each launch creates a new result directory.
- **Expired allocation:** `state.json` can remain `running` after a hard kill. Check
  `qstat -j JOB_ID`, then `qacct -j JOB_ID`. Never signal a recorded PID on another host.
  The stop cell verifies host, job ID and supervisor command. `qdel` ends the entire
  allocation, including Jupyter.
- **No final structures:** inspect `run.log` and `state.json`; checkpoint loading and
  sequence initialization do not establish a completed design.

For unattended batch submission and manual installation repairs, use
[01_running_rfdiffusion_on_scc.ipynb](01_running_rfdiffusion_on_scc.ipynb).
